# 04e — Fine-tuning con CLASS WEIGHTS + más épocas

**Proyecto:** D10Sformer — MIA305 (UdeSA, 2026)  
**Fase:** 4e — Mitigar el colapso de la clase DRAW observado en 4d

## Diagnóstico de 4d

El fine-tune anterior (sin class weights) **colapsó la clase DRAW**:
- Distribución VAL real: home=579, draw=293, away=360
- Predicciones VAL: home=773, **draw=12**, away=447

Resultado: D10Sformer perdió contra LogReg (log_loss 0.88 vs 0.86; ECE 0.040 vs 0.024).

## Hipótesis a testear

Aplicar **class weights inversamente proporcionales a la frecuencia** (recipe sklearn-style):
$$w_i = \frac{n}{k \cdot n_i}$$

Donde $n$ = total samples, $k$ = num classes, $n_i$ = count clase $i$.

Esto **incrementa el costo** del modelo cuando se equivoca en clases minoritarias (DRAW), forzándolo a no colapsar. Es el remedio clásico para imbalanced classification (King & Zeng, 2001).

## Cambios vs 4d

1. **Class weights** en la CE loss del Result head.
2. **15 épocas** (vs 10 antes) — más margen para que las clases minoritarias se acomoden.
3. **Mismo LR (5e-5)** y demás hyperparams.
4. Cargar el `best.pt` del **pre-training** (Fase 4c), no del fine-tune 4d — arrancamos limpio.

## Requisito

Archivo `src/training/trainer.py` actualizado con soporte para `result_class_weights` en `LossSpec`.

---
## 1. Setup

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/d10sformer-v2')
    DATA_ROOT = Path('/content/drive/MyDrive/d10sformer')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_ROOT = PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from paths import ensure_paths, print_paths

paths = ensure_paths(project_root=PROJECT_ROOT, data_root=DATA_ROOT)
print_paths(paths)

# Alias legacy usados en notebooks v1
ROOT = paths.project_root
DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CHECKPOINTS_V1 = paths.checkpoints_v1
DATA_RAW = paths.data_raw
DATA_INTERIM = paths.data_interim


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Necesitamos GPU. Runtime → Change runtime type → T4.')
print(f'torch: {torch.__version__}  |  GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import sys, json, pickle
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader

# paths: ROOT ya definido en setup
# paths: sys.path ya configurado

DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
PRETRAIN_CKPT = paths.checkpoints_v1 / 'pretrain_5ep' / 'best.pt'
assert PRETRAIN_CKPT.exists()

from data.vocabulary import FootballVocab
from data.tokenizer import MatchTokenizer
from data.dataset import MatchDataset
from data.collator import MLMCollator
from models.d10sformer import D10Sformer, D10SformerConfig
from training.trainer import Trainer, TrainerConfig, LossSpec

# Sanity: verificamos que LossSpec ya tiene los campos nuevos
assert 'result_class_weights' in LossSpec.__dataclass_fields__, \
    'Tu trainer.py NO está actualizado. Subí la nueva versión a Drive.'
print('✓ trainer.py actualizado con soporte de class_weights')

---
## 2. Cargar corpus + computar distribución de clases

In [ ]:
vocab = FootballVocab.load(VOCAB_PATH)
tokenizer = MatchTokenizer(vocab, max_seq_length=80)

with open(CORPUS_DIR / 'finetune_train.pkl', 'rb') as f:
    finetune_docs = pickle.load(f)
with open(CORPUS_DIR / 'val.pkl', 'rb') as f:
    val_docs = pickle.load(f)
with open(CORPUS_DIR / 'test.pkl', 'rb') as f:
    test_docs = pickle.load(f)

print(f'Fine-tune train: {len(finetune_docs):,}')
print(f'Val:             {len(val_docs):,}')
print(f'Test:            {len(test_docs):,}')

# Contar clases en train
train_counts = Counter()
for doc in finetune_docs:
    if doc.result is not None:
        train_counts[doc.result] += 1

n_total = sum(train_counts.values())
k = 3  # home_win, draw, away_win
n_home = train_counts['home_win']
n_draw = train_counts['draw']
n_away = train_counts['away_win']

print(f'\nDistribución TRAIN:')
print(f'  home_win: {n_home:5d}  ({100*n_home/n_total:.1f}%)')
print(f'  draw:     {n_draw:5d}  ({100*n_draw/n_total:.1f}%)')
print(f'  away_win: {n_away:5d}  ({100*n_away/n_total:.1f}%)')

# Class weights inversamente proporcionales (sklearn 'balanced')
w_home = n_total / (k * n_home)
w_draw = n_total / (k * n_draw)
w_away = n_total / (k * n_away)

# Orden = [home, draw, away] que es el orden de las heads (RESULT_HOME=0, DRAW=1, AWAY=2)
result_class_weights = [w_home, w_draw, w_away]
print(f'\nClass weights computados:')
print(f'  w_home = {w_home:.3f}')
print(f'  w_draw = {w_draw:.3f}  (mayor peso, clase minoritaria)')
print(f'  w_away = {w_away:.3f}')

---
## 3. Datasets + Dataloaders (igual que 4d)

In [ ]:
ds_train = MatchDataset(finetune_docs, tokenizer)
ds_val   = MatchDataset(val_docs, tokenizer)
ds_test  = MatchDataset(test_docs, tokenizer)

# LabelMappedCollator: convierte target_result_id y target_score_id
# de ids globales del vocab a ids locales de las heads (0..2 / 0..35).
RESULT_VOCAB_TO_LOCAL = {
    vocab.encode('RESULT_HOME_WIN'): 0,
    vocab.encode('RESULT_DRAW'):     1,
    vocab.encode('RESULT_AWAY_WIN'): 2,
}
SCORE_VOCAB_TO_LOCAL = {}
_i = 0
for _h in range(6):
    for _a in range(6):
        SCORE_VOCAB_TO_LOCAL[vocab.encode(f'SCORE_{_h}_{_a}')] = _i
        _i += 1

class LabelMappedCollator:
    def __init__(self, base):
        self.base = base
    def __call__(self, batch):
        b = self.base(batch)
        new_r = b.result_labels.clone()
        for gid, lid in RESULT_VOCAB_TO_LOCAL.items():
            new_r[b.result_labels == gid] = lid
        b.result_labels = new_r
        new_s = b.score_labels.clone()
        for gid, lid in SCORE_VOCAB_TO_LOCAL.items():
            new_s[b.score_labels == gid] = lid
        b.score_labels = new_s
        return b

collator = LabelMappedCollator(MLMCollator(vocab, mlm_probability=0.15, seed=42))

# Sanity check
_t = collator([ds_train[0]])
assert _t.result_labels.max() < 3 and _t.score_labels.max() < 36

BATCH_SIZE = 64
train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collator, num_workers=0)
val_loader   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator, num_workers=0)
test_loader  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator, num_workers=0)

steps_per_epoch = len(train_loader)
print(f'Train batches/época: {steps_per_epoch}')

---
## 4. Modelo + Trainer con CLASS WEIGHTS

In [ ]:
model_config = D10SformerConfig(
    vocab_size=len(vocab),
    d_model=256, num_layers=6, num_heads=8, d_ff=1024,
    max_seq_length=80, num_segments=8,
    dropout=0.1, attention_dropout=0.1,
    pad_token_id=vocab.encode('[PAD]'),
    tie_mlm_weights=True,
)
model = D10Sformer(model_config)

# Arrancamos limpios desde el pre-train (no desde el fine-tune anterior)
ckpt = torch.load(PRETRAIN_CKPT, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f'✓ Cargado pre-train (paso {ckpt["step"]}, val_loss={ckpt["best_val_loss"]:.4f})')

In [ ]:
EPOCHS = 15
MAX_STEPS = steps_per_epoch * EPOCHS
print(f'EPOCHS = {EPOCHS}  |  Pasos totales = {MAX_STEPS}')

trainer_config = TrainerConfig(
    lr=5e-5, weight_decay=0.01, grad_clip_norm=1.0,
    warmup_ratio=0.1,
    max_steps=MAX_STEPS,
    mixed_precision=True,
    log_every=25,
    eval_every=100,
    save_every=400,
    save_best=True,
    output_dir=str(CKPT_DIR),
    run_name='finetune_weighted_15ep',
    seed=42,
)
loss_spec = LossSpec(
    use_mlm=True,    lambda_mlm=0.2,
    use_result=True, lambda_result=1.0,
    use_score=True,  lambda_score=0.3,
    result_class_weights=result_class_weights,   # ← NUEVO
)
print(f'\nLossSpec: λ_mlm={loss_spec.lambda_mlm}, λ_result={loss_spec.lambda_result}, λ_score={loss_spec.lambda_score}')
print(f'Result class weights: {result_class_weights}')

trainer = Trainer(
    model=model, train_loader=train_loader, val_loader=val_loader,
    config=trainer_config, loss_spec=loss_spec,
)
print(f'\nDevice: {trainer.device}  |  AMP: {trainer.use_amp}')
print(f'Result weights tensor: {trainer._result_weights_tensor}')

---
## 5. Fine-tuning (15 épocas con class weights)

In [ ]:
import time
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f'\n✓ Fine-tuning completado en {elapsed/60:.1f} min ({MAX_STEPS/elapsed:.1f} step/s)')
print(f'  Best val_loss: {trainer.best_val_loss:.4f}')

---
## 6. Evaluación detallada sobre VAL y TEST

In [ ]:
from eval.metrics import evaluate_all

best_path = trainer.output_dir / 'best.pt'
best_ckpt = torch.load(best_path, map_location=trainer.device, weights_only=False)
model.load_state_dict(best_ckpt['model_state_dict'])
model.eval()
print(f'✓ Cargado best (step={best_ckpt["step"]}, val_loss={best_ckpt["best_val_loss"]:.4f})')

@torch.no_grad()
def predict_result_probs(loader):
    all_probs, all_true = [], []
    for batch in loader:
        batch = batch.to(trainer.device)
        out = model(batch.token_ids, batch.segment_ids, attention_mask=batch.attention_mask)
        probs = F.softmax(out['result_logits'], dim=-1).cpu().numpy()
        labels = batch.result_labels.cpu().numpy()
        for p, y in zip(probs, labels):
            if y == -100: continue
            all_probs.append(p); all_true.append(int(y))
    return np.array(all_true), np.array(all_probs)

y_val,  p_val  = predict_result_probs(val_loader)
y_test, p_test = predict_result_probs(test_loader)

metrics_val  = evaluate_all(y_val,  p_val)
metrics_test = evaluate_all(y_test, p_test)
print('\n=== D10Sformer (4e, con class weights) ===')
print(f'{"métrica":<20} {"VAL":<12} {"TEST":<12}')
for k in ['log_loss', 'brier', 'ece', 'accuracy']:
    print(f'{k:<20} {metrics_val[k]:<12.4f} {metrics_test[k]:<12.4f}')

In [ ]:
# Confusion matrix: ¿ahora predice DRAW?
preds_val = p_val.argmax(axis=1)
cm = np.zeros((3, 3), dtype=int)
for yt, yp in zip(y_val, preds_val):
    cm[yt, yp] += 1

labels = ['home_win', 'draw', 'away_win']
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() * 0.6 else 'black')
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_yticks(range(3)); ax.set_yticklabels(labels)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion matrix (VAL, n={len(y_val)}) — con class weights')
plt.colorbar(im); plt.tight_layout(); plt.show()

print(f'\nDistribución VAL: {Counter(y_val.tolist())}')
print(f'Predicciones VAL: {Counter(preds_val.tolist())}')
print(f'\nDraws predichos: {(preds_val == 1).sum()} de {len(y_val)} ({100*(preds_val == 1).sum()/len(y_val):.1f}%)')

---
## 7. Comparación final: baselines vs 4d (sin weights) vs 4e (con weights)

In [ ]:
# Baselines de Fase 1 (test)
baselines = {
    'LogReg':            {'log_loss': 0.8610, 'brier': 0.5071, 'ece': 0.0236, 'accuracy': 0.6005},
    'XGBoost':           {'log_loss': 0.8663, 'brier': 0.5105, 'ece': 0.0182, 'accuracy': 0.6026},
    'LightGBM':          {'log_loss': 0.8744, 'brier': 0.5135, 'ece': 0.0250, 'accuracy': 0.5990},
    'ELO solo':          {'log_loss': 1.0102, 'brier': 0.6101, 'ece': 0.0890, 'accuracy': 0.5510},
    'D10Sformer 4d (sin weights, test)':  {'log_loss': 0.8846, 'brier': 0.5221, 'ece': 0.0400, 'accuracy': 0.5912},
    'D10Sformer 4e (con weights, val)':   {k: metrics_val[k]  for k in ['log_loss','brier','ece','accuracy']},
    'D10Sformer 4e (con weights, test)':  {k: metrics_test[k] for k in ['log_loss','brier','ece','accuracy']},
}

import pandas as pd
df = pd.DataFrame(baselines).T
print(df.to_string(float_format=lambda x: f'{x:.4f}'))

best_baseline_ll  = min(baselines[k]['log_loss'] for k in ['LogReg','XGBoost','LightGBM'])
best_baseline_acc = max(baselines[k]['accuracy'] for k in ['LogReg','XGBoost','LightGBM'])
best_baseline_ece = min(baselines[k]['ece']      for k in ['LogReg','XGBoost','LightGBM'])

def cmp(v, ref, lower_better=True):
    if lower_better:
        return f'{"✓ GANA" if v < ref else "✗ pierde"} (Δ = {v - ref:+.4f})'
    return f'{"✓ GANA" if v > ref else "✗ pierde"} (Δ = {v - ref:+.4f})'

print(f'\n--- D10Sformer 4e (test) vs mejor baseline ---')
print(f'log_loss:  {metrics_test["log_loss"]:.4f}  vs  {best_baseline_ll:.4f}  →  {cmp(metrics_test["log_loss"], best_baseline_ll)}')
print(f'accuracy:  {metrics_test["accuracy"]:.4f}  vs  {best_baseline_acc:.4f}  →  {cmp(metrics_test["accuracy"], best_baseline_acc, lower_better=False)}')
print(f'ece:       {metrics_test["ece"]:.4f}  vs  {best_baseline_ece:.4f}  →  {cmp(metrics_test["ece"], best_baseline_ece)}')

---
## 8. Conclusiones de Fase 4e

Llenar al final:

- [ ] Draws predichos VAL: _____ de 293 reales (4d eran 12)
- [ ] D10Sformer 4e test log_loss: _____  vs LogReg 0.8610
- [ ] D10Sformer 4e test accuracy: _____  vs XGBoost 0.6026
- [ ] D10Sformer 4e test ECE: _____  vs XGBoost 0.0182
- [ ] ¿Ganamos en alguna métrica? _____
- [ ] ¿La confusion matrix se ve balanceada? _____

**Decisión próxima:**
- Si **ganamos en log_loss o ECE** → tenemos paper. Vamos a Fase 5 (eval profunda + reliability) y Fase 6 (Monte Carlo).
- Si seguimos perdiendo pero estamos cerca (Δ < 0.01) → ablations adicionales: ELO continuo en lugar de bucketizado, sin MLM aux.
- Si seguimos perdiendo por mucho → reportamos honestamente y usamos baselines para el Mundial.